In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split
from src.utils import * 
import src.prompt as prompt
from src.data_loader import load_spatial_data_csv



In [ ]:
representetive_gene_list = (repository_root() / "examples/starmap/representative_genes.txt").read_text().splitlines()


In [ ]:
import openai
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# data

In [ ]:
config = load_config("configs/config_finetunePro_starmap.yaml")

# in case you want to change the data name, replicate, folder path, output path
config.data_name = "BZ5"
# IMPORTANT: check replicate"
config.replicate = "_localfinetune1" 

config.refresh_paths()



In [ ]:
data_path = str(dataset_dir("starmap", config.data_name))

# Load data using the new function
adata = load_spatial_data_csv(
    data_path=data_path,
    main_data_file="data.csv",
    celltype_file="celltype.csv", 
    pos_file="pos.csv",
    domain_file="domain.csv",
    config=config,
    index_col=0,
    first_column_names=True
)

In [ ]:
# rename the domain, the name is useful in sample prompt
domain_mapping = {1 : "Layer 1", 2 : "Layer 2/3", 3 : "Layer 5", 4 : "Layer 6"}
adata.obs[config.name_truth] = adata.obs[config.name_truth].map(domain_mapping)

In [ ]:

# Prepare neighbor data using the new function
neighbor_normalized_df, neighbor_normalized_df_genes, adj_matrix = prepare_neighbor_data(
    adata, config, representetive_gene_list
)

## sample data

In [ ]:
# 设定随机种子
seed = 42  # 你可以根据需要修改这个值

# 定义分割比例 p (比如 0.7 表示 70% 数据用于训练，30% 数据用于测试)
p = 0.3

# 分割数据集为训练集和测试集
train_neighbor_normalized_df, val_neighbor_normalized_df = train_test_split(neighbor_normalized_df, 
                                                            test_size=1-p, 
                                                            random_state=seed,
                                                            stratify=adata.obs[config.name_truth]
                                                           )



In [ ]:
# check sample distribution
adata.obs.loc[train_neighbor_normalized_df.index, config.name_truth].value_counts()

In [ ]:
# # finetune train and val data
# _, val_for_finetune = train_test_split(val_neighbor_normalized_df, 
#                                                             test_size=0.1, 
#                                                             random_state=seed
#                                                            )

# adata.obs.loc[val_for_finetune.index, config.name_truth].value_counts()

In [ ]:
train_neighbor_df = train_neighbor_normalized_df
one_shot_df = pd.concat([adata.obs[config.name_truth].loc[train_neighbor_df.index], train_neighbor_df], axis=1).groupby(config.name_truth, observed=False).mean()
print(one_shot_df.index)

# prompt

In [ ]:
# domain_mapping = {1 : "Layer 1", 2 : "Layer 2/3", 3 : "Layer 5", 4 : "Layer 6"}

cell_names_mapping = {'Astro': 'Astrocytes',
 'Endo': 'Endothelial cells',
 'L5-1': 'Layer 5 pyramidal neuron subtype 1',
 'Lhx6': 'Lhx6-expressing interneurons',
 'NPY': 'Neuropeptide Y-expressing interneurons',
 'Oligo': 'Oligodendrocytes',
 'Reln': 'Reelin-expressing cells',
 'SST': 'Somatostatin-expressing interneurons',
 'Smc': 'Smooth muscle cells',
 'VIP': 'Vasoactive intestinal peptide-expressing interneurons',
 'eL2/3': 'Excitatory neuron layer 2/3',
 'eL5-2': 'Excitatory neuron layer 5 subtype 2',
 'eL5-3': 'Excitatory neuron layer 5 subtype 3',
 'eL6-1': 'Excitatory neuron layer 6 subtype 1',
 'eL6-2': 'Excitatory neuron layer 6 subtype 2'}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping
config.system_prompt = prompt.CP_celltype(one_shot_df,config)



In [ ]:
print(config.system_prompt)

# GPT-4o-mini

In [ ]:
# generate json for finetune
output_folder = f"finetune_json/{config.data_name}_{config.model_type}/"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

train_output_file = f"{output_folder}{config.data_name}_train_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
print(f"Generating json for training into {train_output_file}")
with open(train_output_file, 'w') as f:
    for i in range(train_neighbor_normalized_df.shape[0]):
        system_p = config.system_prompt
        user_p = prompt.finetune_user_celltype(train_neighbor_normalized_df, i, config)
        assistant_p = prompt.finetune_assistant(train_neighbor_normalized_df, i, adata.obs[config.name_truth])
        
        row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
        json_str = json.dumps(row_data)
        f.write(json_str + '\n')  

# val_output_file = f"{output_folder}{config.data_name}_val_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
# print(f"Generating json for validation into {val_output_file}")
# with open(val_output_file, 'w') as f:
#     for i in range(val_for_finetune.shape[0]):
#         system_p = config.system_prompt
#         user_p = prompt.finetune_user_celltype(val_for_finetune, i, config)
#         assistant_p = prompt.finetune_assistant(val_for_finetune, i, adata.obs[config.name_truth])

#         row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
#         json_str = json.dumps(row_data)
#         f.write(json_str + '\n')  

In [ ]:
print(system_p + user_p + assistant_p)

In [ ]:
# finetune with openai gpt-4o-mini
# train_file = client.files.create(
#   file=open(train_output_file, "rb"),
#   purpose="fine-tune"
# )

# # val_file = client.files.create(
# #   file=open(val_output_file, "rb"),
# #   purpose="fine-tune"
# # )

# finetune_job = client.fine_tuning.jobs.create(
#   training_file=train_file.id,
#   # validation_file=val_file.id,
#   model="gpt-4o-mini-2024-07-18",
#   suffix=f"{config.model_type}_{config.r}"   # default is personal
# )


# gemini 1.5 flash

In [ ]:
import google

In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.environ["API_KEY"])
for model_info in genai.list_tuned_models():
    print(model_info.name)

In [ ]:

tunable_models = [
    m for m in genai.list_models()
    if "createTunedModel" in m.supported_generation_methods]
tunable_models

In [ ]:
base_model = "models/gemini-1.5-flash-001-tuning"
training_data = []
for i in range(train_neighbor_normalized_df.shape[0]):
    system_p = config.system_prompt
    user_p = prompt.finetune_user_celltype(train_neighbor_normalized_df, i, config)
    assistant_p = prompt.finetune_assistant(train_neighbor_normalized_df, i, adata.obs[config.name_truth])
    training_data.append({'text_input': system_p + user_p, 'output': assistant_p})
    



In [ ]:
training_data[:4]

In [ ]:
operation = genai.create_tuned_model(
    # You can use a tuned model here too. Set `source_model="tunedModels/..."`
    display_name=config.data_name,
    source_model=base_model,
    epoch_count=30,
    batch_size=4,
    learning_rate=0.001,
    training_data=training_data,
)

In [ ]:
for status in operation.wait_bar():
    time.sleep(10)

In [ ]:
result = operation.result()

In [ ]:
import seaborn as sns
model = operation.result()  # model = genai.get_tuned_model()
snapshots = pd.DataFrame(model.tuning_task.snapshots)

sns.lineplot(data=snapshots, x = 'epoch', y='mean_loss')

# Use the finetuned model

## generate json

In [ ]:
# add model name to config

# reload config
config = load_config("configs/config_finetunePro_starmap.yaml")
domain_mapping = {1 : "Layer 1", 2 : "Layer 2/3", 3 : "Layer 5", 4 : "Layer 6"}

cell_names_mapping = {'Astro': 'Astrocytes',
 'Endo': 'Endothelial cells',
 'L5-1': 'Layer 5 pyramidal neuron subtype 1',
 'Lhx6': 'Lhx6-expressing interneurons',
 'NPY': 'Neuropeptide Y-expressing interneurons',
 'Oligo': 'Oligodendrocytes',
 'Reln': 'Reelin-expressing cells',
 'SST': 'Somatostatin-expressing interneurons',
 'Smc': 'Smooth muscle cells',
 'VIP': 'Vasoactive intestinal peptide-expressing interneurons',
 'eL2/3': 'Excitatory neuron layer 2/3',
 'eL5-2': 'Excitatory neuron layer 5 subtype 2',
 'eL5-3': 'Excitatory neuron layer 5 subtype 3',
 'eL6-1': 'Excitatory neuron layer 6 subtype 1',
 'eL6-2': 'Excitatory neuron layer 6 subtype 2'}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping
config.system_prompt = prompt.CP_celltype(one_shot_df,config)




In [ ]:
print(config.gpt_model)
print(config.data_name)
config.replicate = "_rep2R700"
print(config.replicate)

In [ ]:
generate_json_end2end(val_neighbor_normalized_df, config, prompt_func=prompt.finetune_user_celltype, n_rows=1, batch_size=5000)

## submit

In [ ]:

import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_finetunePro_starmap.yaml {config.data_name} {config.replicate} > outs/{config.data_name}_{config.model_type}_{config.replicate}.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 1
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']

                # extract outputs - handle both JSON format and text format
                extract_dict = extract_json_microenvironment(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['finetunePro_gpt4o_mini']

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
gpt_results_df.finetunePro_gpt4o_mini.value_counts()

In [ ]:
gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == "Layer2/ 3", "finetunePro_gpt4o_mini"] = "Layer2/3"

In [ ]:
# fill nan with unknown
gpt_results_df.fillna("unknown", inplace=True)
# if the number of the cell type is less than 4, set it to unknown
for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<4].index:
    gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = "unknown"

In [ ]:
# # manually retrieve batch output
# output_file_name = f"{output_path}/response_BZ5_1_{use_full_name}_{with_self_type}_{with_region_name}_{Graph_type}_{with_negatives}_{with_CoT}_{with_count_numbers}.txt"
# batch_id = "batch_66f62020d11c81909424c1423da11a70"
# file_response = client.files.content(client.batches.retrieve(batch_id).output_file_id)
# # Open the file in write mode and save the string
# with open(output_file_name, 'w') as file:
#     file.write(file_response.text)

## plot and save

In [ ]:
val_adata = adata[val_neighbor_normalized_df.index].copy()
val_adata.obs = val_adata.obs.join(gpt_results_df)
sc.pl.scatter(val_adata, x="x", y="y", color="finetunePro_gpt4o_mini", title =  f"finetunePro_gpt4o_mini")

print(adjusted_rand_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini']))
print(normalized_mutual_info_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini']))

In [ ]:
gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")



In [ ]:
# refine the niche
pos_data = adata.obs[config.pos_name]
val_adj_matrix, _ = sparse_adjacency(pos_data.loc[val_neighbor_normalized_df.index], threshold=config.r)

refined_niche = relabel_cells(val_adj_matrix.toarray(), val_adata.obs['finetunePro_gpt4o_mini'])
val_adata.obs['finetunePro_gpt4o_mini_refined'] = refined_niche
sc.pl.scatter(val_adata, x="x", y="y", color="finetunePro_gpt4o_mini_refined", title =  f"finetunePro_gpt4o_mini_refined")
print(adjusted_rand_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini_refined']))
print(normalized_mutual_info_score(val_adata.obs[config.name_truth], val_adata.obs['finetunePro_gpt4o_mini_refined']))


In [ ]:
print(f"save to ./gpt4omini_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
val_adata.obs.to_csv(f"./gpt4omini_results/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

# test

## load test data BZ9 BZ14 
Prototype in prompt is based on training data

In [ ]:
config = load_config("configs/config_finetunePro_starmap.yaml")
config.data_name = "BZ9"
config.replicate = "_rep1R700"
config.refresh_paths()


domain_mapping = {1 : "Layer 1", 2 : "Layer 2/3", 3 : "Layer 5", 4 : "Layer 6"}
cell_names_mapping = {'Astro': 'Astrocytes',
 'Endo': 'Endothelial cells',
 'L5-1': 'Layer 5 pyramidal neuron subtype 1',
 'Lhx6': 'Lhx6-expressing interneurons',
 'NPY': 'Neuropeptide Y-expressing interneurons',
 'Oligo': 'Oligodendrocytes',
 'Reln': 'Reelin-expressing cells',
 'SST': 'Somatostatin-expressing interneurons',
 'Smc': 'Smooth muscle cells',
 'VIP': 'Vasoactive intestinal peptide-expressing interneurons',
 'eL2/3': 'Excitatory neuron layer 2/3',
 'eL5-2': 'Excitatory neuron layer 5 subtype 2',
 'eL5-3': 'Excitatory neuron layer 5 subtype 3',
 'eL6-1': 'Excitatory neuron layer 6 subtype 1',
 'eL6-2': 'Excitatory neuron layer 6 subtype 2'}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping
config.system_prompt = prompt.CP_celltype(one_shot_df,config)


In [ ]:
# --- Load data ---
data_path = str(dataset_dir("starmap", config.data_name))
x_data_name = "data.csv"  
index_col = 0
adata = sc.read_csv(f"{data_path}/{x_data_name}", first_column_names=True)
celltype_data = pd.read_csv(f"{data_path}/celltype.csv", index_col=index_col)  # Assuming first column is index
celltype_data.columns = ["cell_type"] 
pos_data = pd.read_csv(f"{data_path}/pos.csv", index_col=index_col)
pos_data.columns = ['x', 'y']
domain_data = pd.read_csv(f"{data_path}/domain.csv", index_col=index_col)
domain_data.columns = [config.name_truth]
adata.obs = adata.obs.join([celltype_data, pos_data, domain_data])

# clean the cell ID to save token
adata.obs_names = list(range(len(adata)))
adata.obs_names = adata.obs_names.astype(str)

# rename the domain
domain_mapping = {1 : "Layer 1", 2 : "Layer 2/3", 3 : "Layer 5", 4 : "Layer 6"}
adata.obs[config.name_truth] = adata.obs[config.name_truth].map(domain_mapping)

pos_data = adata.obs[['x', 'y']]
celltype_data = adata.obs[['cell_type']]
domain_data = adata.obs[[config.name_truth]]

# --- Compute adjacency matrix ---
r = config.r
adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
# add diagonal to the adj_matrix
adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

# --- Generate one-hot encoded matrix ---
one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
one_hot_matrix = one_hot_df.values
one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(one_hot_matrix)
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized.toarray(), 
                              index=celltype_data.index, 
                              columns=one_hot_df.columns.str.lstrip('_'))


## test GPT

In [ ]:
generate_json_end2end(neighbor_normalized_df, config, prompt_func=prompt.finetune_user_celltype, n_rows=1, batch_size=5000)

In [ ]:
# submit_end2end.py
# nohup python -u -m src.submit_end2end configs/config_BZ9_zeroshot.yaml > BZ9_zeroshot.out 2>&1 &
import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_finetunePro_starmap.yaml {config.data_name} {config.replicate} > outs/{config.data_name}_finetunePro_BZ5.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 1
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ").replace("‘", "'").replace("’", "'")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['finetunePro_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
gpt_results_df.value_counts()

In [ ]:
# fill nan with unknown
gpt_results_df.fillna("unknown", inplace=True)
# if the number of the cell type is less than 4, set it to unknown
for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<4].index:
    gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = "unknown"

In [ ]:
adata.obs = adata.obs.join(gpt_results_df)
# replace the NA in adata.obs['finetunePro_gpt4o'] with "unknown"
adata.obs['finetunePro_gpt4o_mini'] = adata.obs['finetunePro_gpt4o_mini'].fillna("unknown")

sc.pl.scatter(adata, x="x", y="y", color="finetunePro_gpt4o_mini", title =  f"finetunePro_gpt4o_mini")
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini']))
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini']))

## save results

In [ ]:
# gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
# print(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


## refine

In [ ]:
# refine the niche
config.r_factor = 1
refined_niche = relabel_cells(adj_matrix.toarray(), gpt_results_df['finetunePro_gpt4o_mini'])
adata.obs['finetunePro_gpt4o_mini_refined'] = refined_niche
# sc.pl.spatial(adata, color="finetunePro_gpt4o_mini_refined", library_id=config.data_name, title =  f"finetunePro_gpt4o_mini_refined")
# print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_refined']))
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_refined']))


In [ ]:
save_folder = "./finetune_results/STARmap"
adata.obs.to_csv(f"{save_folder}/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

print(f"save first stage result to {save_folder}/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


# load first stage result

# high confidence cell

In [ ]:
label_key = 'finetunePro_gpt4o_mini'  # IMPORTANT!!!!! before refinement
high_confidence_mask = find_high_confidence_cells(
    adata,
    pos_data = pos_data,
    label_key=label_key,
    k=min(30, int((np.mean(n_neighbors))/2)),  # Consider top 20/ half of the mean nearest neighbors  # IMPORTANT!!!!!!  others are min(20) only merfish29 is min(30)
    distance_threshold=config.r
)

adata.obs['confident_cells'] = adata.obs['finetunePro_gpt4o_mini'].copy()
adata.obs['confident_cells'] = adata.obs['confident_cells'].astype(str)
adata.obs.loc[~high_confidence_mask, 'confident_cells'] = 'unconfident'


In [ ]:
sc.pl.scatter(adata, x="x", y="y", color="confident_cells", title =  f"confident_cells")

In [ ]:
# Get unique values excluding 'unconfident'
confident_unique = set(adata.obs.loc[high_confidence_mask, 'confident_cells'].unique()) - {'unconfident'}
unique_layers = set(adata.obs[label_key].unique()) - {'unknown'}

# Check if any elements are missing
missing_elements = unique_layers - confident_unique
old_one_shot_df = pd.DataFrame()
if len(missing_elements) > 0:
    print(f"Missing elements in confident cells: {missing_elements}")
    print("use the old one_shot_df for that niche")
    old_one_shot_df = one_shot_df.loc[list(missing_elements)]
    
else:
    print("No missing elements in confident cells") 

conserved_normalized_df = neighbor_normalized_df.loc[high_confidence_mask]
# conserved_normalized_df_genes = neighbor_normalized_df_genes.loc[high_confidence_mask]

un_conserved_normalized_df = neighbor_normalized_df.loc[~high_confidence_mask]
# un_conserved_normalized_df_genes = neighbor_normalized_df_genes.loc[~high_confidence_mask]   

print(f"find {len(conserved_normalized_df)} conserved cells")
print(f"find {len(un_conserved_normalized_df)} un-conserved cells")

# calculate prototype
conserved_neighbor_df = conserved_normalized_df.copy()
one_shot_df = pd.concat([adata.obs[label_key].loc[conserved_neighbor_df.index], conserved_neighbor_df], axis=1).groupby(label_key, observed=False).mean()
one_shot_df = one_shot_df.dropna()

# remove unknown
if 'unknown' in one_shot_df.index:
    one_shot_df = one_shot_df.drop(index='unknown')
if len(old_one_shot_df) > 0:
    one_shot_df = pd.concat([old_one_shot_df, one_shot_df], axis=0)
# update config.domain_mapping
print("old one_shot_df.index", config.domain_mapping)
print("new one_shot_df.index", one_shot_df.index)
print("update config.domain_mapping")
config.domain_mapping = {i+1: one_shot_df.index[i] for i in range(len(one_shot_df))}
print("new config.domain_mapping", config.domain_mapping)

config.system_prompt = prompt.CP_celltype(one_shot_df,config)
if config.with_negatives:
    # get top 3 genes for each row in one_shot_df
    neg_top_3_genes = one_shot_df[config.gene_names].apply(lambda x: x.nlargest(3).index.tolist(), axis=1)
    config.neg_top_3_genes = neg_top_3_genes

print(config.folder_path)
print(config.gpt_model)
config.replicate = "_rep1R700_unconserved"  # "_rep2R100_unconserved" is for one shot, _rep2R100_finetune is for finetune
print(config.replicate)


In [ ]:
# json is from un-conserved cells
generate_json_end2end(un_conserved_normalized_df, config, prompt_func=prompt.finetune_user_celltype, max_completion_tokens=128, batch_size=3000)


In [ ]:
# submit_end2end.py

import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_finetunePro_starmap.yaml {config.data_name} {config.replicate} > outs/{config.data_name}_{config.model_type}_{config.replicate}.out 2>&1 &"
# cmd = f"python -u -m src.submit_end2end configs/config_finetunePro_starmap.yaml {config.data_name} {config.replicate} > outs/{config.data_name}_{config.model_type}_{config.replicate}.out 2>&1"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
gpt_results_df = pd.DataFrame()
if len(un_conserved_normalized_df) > 3000:
    n_batch = 2
else:
    n_batch = 1

for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['finetunePro_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)
gpt_results_df.index = gpt_results_df.index.str.replace("id_", "")


In [ ]:
# fill nan with unknown
gpt_results_df.fillna("unknown", inplace=True)
# if the number of the cell type is less than 4, set it to unknown
for nichtype in gpt_results_df.finetunePro_gpt4o_mini.value_counts()[gpt_results_df.finetunePro_gpt4o_mini.value_counts()<4].index:
    gpt_results_df.loc[gpt_results_df.finetunePro_gpt4o_mini == nichtype, "finetunePro_gpt4o_mini"] = "unknown"


In [ ]:
# update the finetune_gpt4o_mini with second stage results
adata.obs['finetunePro_gpt4o_mini_twostage'] = adata.obs[label_key].copy()
adata.obs['finetunePro_gpt4o_mini_twostage'] = adata.obs['finetunePro_gpt4o_mini_twostage'].astype(str)
adata.obs.loc[gpt_results_df.index, 'finetunePro_gpt4o_mini_twostage'] = gpt_results_df.finetunePro_gpt4o_mini
# sc.pl.spatial(adata, color="finetunePro_gpt4o_mini_twostage", library_id=config.data_name, title =  f"finetunePro_gpt4o_mini_twostage")
# print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_twostage']))
# print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_twostage']))
# refine the niche
adj_matrix, _ = sparse_adjacency(pos_data.loc[neighbor_normalized_df.index], threshold=r)

refined_niche = relabel_cells(adj_matrix.toarray(), adata.obs['finetunePro_gpt4o_mini_twostage'])
adata.obs['finetunePro_gpt4o_mini_twostage_refined'] = refined_niche
# sc.pl.spatial(adata, color="finetunePro_gpt4o_mini_twostage_refined", library_id=config.data_name, title =  f"finetunePro_gpt4o_mini_twostage_refined")
# print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_twostage_refined']))
print(normalized_mutual_info_score(adata.obs[config.name_truth], adata.obs['finetunePro_gpt4o_mini_twostage_refined']))


In [ ]:
save_folder = './twostage_results/STARmap'
adata.obs.to_csv(f"{save_folder}/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
print(f"save second stage result to {save_folder}/{config.data_name}_with_refined_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
